In [1]:
import numpy as np
import pandas as pd

# VN30

In [2]:
df_vn30 = pd.read_csv("./vn30_from_2017_to_2025.csv")
df_vn30 = df_vn30[["symbol", "time", "close", "volume"]]

df_vn30['time'] = pd.to_datetime(df_vn30['time'])

df_vn30.head()

,symbol,time,close,volume
0,ACB,2017-01-03,3.33,1609757
1,ACB,2017-01-04,3.35,739000
2,ACB,2017-01-05,3.35,471881
3,ACB,2017-01-06,3.53,1409824
4,ACB,2017-01-09,3.65,970791


In [3]:
df_vn30.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62196 entries, 0 to 62195
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   symbol  62196 non-null  object        
 1   time    62196 non-null  datetime64[ns]
 2   close   62196 non-null  float64       
 3   volume  62196 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 1.9+ MB


# VN INDEX

In [4]:
df_vnindex = pd.read_csv("./vnindex_from_2017_to_2025.csv")
df_vnindex = df_vnindex[["Ngày", "Lần cuối"]]
df_vnindex = df_vnindex.rename(columns={
    'Ngày': 'time',
    'Lần cuối': 'vnindex',
})

df_vnindex['time'] = pd.to_datetime(df_vnindex['time'], dayfirst=True)

df_vnindex['vnindex'] = (
    df_vnindex['vnindex']
    .str.replace(',', '', regex=False)
    .astype(float)
)

df_vnindex.head()

,time,vnindex
0,2025-12-31,1784.49
1,2025-12-30,1766.90
2,2025-12-29,1754.84
3,2025-12-26,1729.80
4,2025-12-25,1742.85


In [5]:
df_vnindex.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2248 entries, 0 to 2247
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   time     2248 non-null   datetime64[ns]
 1   vnindex  2248 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 35.2 KB


# Join

In [6]:
# time có ở vn30 nhưng không có ở vnindex
missing_in_vnindex = df_vn30[
    ~df_vn30['time'].isin(df_vnindex['time'])
]

print(missing_in_vnindex.shape)
missing_in_vnindex

(24, 4)


,symbol,time,close,volume
266,ACB,2018-01-24,7.21,4139581
2510,BID,2018-01-24,15.53,0
4759,CTG,2018-01-24,11.78,0
7008,DGC,2018-01-24,6.98,54600
9251,FPT,2018-01-24,15.93,0
11500,GAS,2018-01-24,60.29,0
15438,HDB,2018-01-24,8.63,0
17687,HPG,2018-01-24,12.05,0
19748,LPB,2018-01-24,5.62,2824150
21987,MBB,2018-01-24,6.17,0


In [7]:
# time có ở vnindex nhưng không có ở vn30
missing_in_vn30 = df_vnindex[
    ~df_vnindex['time'].isin(df_vn30['time'])
]

print(missing_in_vn30.shape)
missing_in_vn30

(0, 2)


,time,vnindex


In [8]:
df = df_vn30.merge(
    df_vnindex,
    on='time',
    how='inner',        
)

df.head()

,symbol,time,close,volume,vnindex
0,ACB,2017-01-03,3.33,1609757,672.01
1,ACB,2017-01-04,3.35,739000,674.70
2,ACB,2017-01-05,3.35,471881,675.81
3,ACB,2017-01-06,3.53,1409824,679.80
4,ACB,2017-01-09,3.65,970791,682.57


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62172 entries, 0 to 62171
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   symbol   62172 non-null  object        
 1   time     62172 non-null  datetime64[ns]
 2   close    62172 non-null  float64       
 3   volume   62172 non-null  int64         
 4   vnindex  62172 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 2.4+ MB


In [10]:
df.describe()

,time,close,volume,vnindex
count,62172,62172.000000,6.217200e+04,62172.000000
mean,2021-09-10 06:55:50.550086912,34.432159,5.618873e+06,1130.065015
min,2017-01-03 00:00:00,1.390000,0.000000e+00,659.210000
25%,2019-07-16 00:00:00,11.670000,8.727900e+05,962.160000
50%,2021-09-27 00:00:00,23.500000,2.283018e+06,1111.180000
75%,2023-11-16 00:00:00,50.142500,6.295367e+06,1273.110000
max,2025-12-31 00:00:00,219.100000,2.497607e+08,1784.490000
std,NaN,30.029118,9.723886e+06,233.121958


In [11]:
df.symbol.nunique()

30

In [12]:
df.groupby('symbol').agg(
    time_min=('time', 'min'),
    time_max=('time', 'max'),
    count=('time', 'size')
)

,time_min,time_max,count
symbol,,,
ACB,2017-01-03,2025-12-31,2243
BID,2017-01-03,2025-12-31,2248
CTG,2017-01-03,2025-12-31,2248
DGC,2017-01-03,2025-12-31,2242
FPT,2017-01-03,2025-12-31,2248
GAS,2017-01-03,2025-12-31,2248
GVR,2018-03-21,2025-12-31,1942
HDB,2018-01-05,2025-12-31,1995
HPG,2017-01-03,2025-12-31,2248


In [13]:
min_len = 2000

valid_symbols = (
    df.groupby('symbol').size()
    .loc[lambda x: x >= min_len]
    .index
)

valid_symbols

Index(['ACB', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'HPG', 'LPB', 'MBB', 'MSN',
       'MWG', 'PLX', 'SAB', 'SHB', 'SSI', 'STB', 'VCB', 'VHM', 'VIB', 'VIC',
       'VJC', 'VNM', 'VPB', 'VRE'],
      dtype='object', name='symbol')

In [14]:
df_filtered = df[df['symbol'].isin(valid_symbols)]
df_filtered.symbol.nunique()

24

In [15]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53050 entries, 0 to 62171
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   symbol   53050 non-null  object        
 1   time     53050 non-null  datetime64[ns]
 2   close    53050 non-null  float64       
 3   volume   53050 non-null  int64         
 4   vnindex  53050 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 2.4+ MB


In [16]:
df_filtered.to_csv('vn30_filtered.csv', index=False)